# 003 - Модуль построения target dataset

Notebook вокруг `research/target_building/build_targets_dataset.py`.

Структура:
1. полный исходный код модуля,
2. короткое объяснение ключевых инженерных решений,
3. пример вызова: `parquet -> targets parquet`.


In [4]:
from pathlib import Path


# Ищем корень репозитория независимо от того, из какой папки запущен notebook.
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "research").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень репозитория с папками research/ и notebooks/")


ROOT = find_repo_root(Path.cwd())
MODULE_PATH = ROOT / "research" / "target_building" / "build_targets_dataset.py"
print(MODULE_PATH)
print(MODULE_PATH.read_text(encoding="utf-8"))


D:\tumar\okx-hft-research\research\target_building\build_targets_dataset.py
# flake8: noqa
"""Build time-based ML target datasets from reconstructed order book data.

This module focuses on target construction only:
- time-based horizon windows using real `ts_event`,
- symmetric and asymmetric first-hit labels,
- MFE / MAE-style excursion targets in ticks,
- compact summary helpers,
- CLI/demo entrypoint for parquet -> parquet workflows.

The implementation deliberately avoids feature engineering so it can later be
composed with a separate `build_features(...)` stage.
"""

from __future__ import annotations

import argparse
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from research.utils.tick_size import infer_tick_size

LOGGER = logging.getLogger(__name__)

BASE_ID_COLUMNS = [
    "ts_event",
    "inst_id",
    "anchor_snapshot_ts",
    "reconstruction_mode",
    "mid_px",
    "spread_px",
]



## Ключевые решения

- Окна строятся по реальному времени `ts_event` через `np.searchsorted`, а не по фиксированному числу строк.
- Код отвечает только за таргеты: feature engineering сюда не подмешан.
- Основная логика работает на массивах NumPy, чтобы не делать лишние `DataFrame`-срезы внутри цикла по строкам.
- Симметричные метки и асимметричные TP/SL-подобные метки разделены явно, чтобы позже было легко расширить код до meta-labeling или maker-entry assumptions.
- `summarize_targets(...)` возвращает компактные таблицы, а не печатает разрозненный notebook-style вывод.
- В модуле есть блок `__main__` в CLI-стиле для воспроизводимого сценария `parquet -> parquet`.


In [5]:
from pathlib import Path

import pandas as pd

from research.target_building.build_targets_dataset import build_targets_dataset, summarize_targets
from research.utils.tick_size import infer_tick_size


# Ищем корень репозитория независимо от текущей рабочей директории kernel.
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "research").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень репозитория с папками research/ и notebooks/")


ROOT = find_repo_root(Path.cwd())
PARQUET_PATH = ROOT / "data" / "processed" / "modeling" / "btc_usdt_swap_model_dataset_100ms_baseline_20260323_000000__20260325_235959.parquet"
OUTPUT_PATH = ROOT / "outputs" / "targets" / "btc_usdt_swap_targets_dataset.parquet"

# Основные параметры построения таргетов.
HORIZONS_SEC = [30, 60, 120, 180]
THRESHOLDS_TICKS = [50, 100, 150, 200, 260, 300]
ADVERSE_TICKS = [25, 50, 75, 100, 120, 150]
SAMPLING_STEP_ROWS = 1
DROP_LAST_INCOMPLETE = True

# Загружаем parquet и автоматически определяем tick size по mid_px.
df = pd.read_parquet(PARQUET_PATH)
tick_size = infer_tick_size(df, price_col="mid_px").tick_size

# Строим чистый dataset с target-колонками.
df_targets = build_targets_dataset(
    df,
    tick_size=tick_size,
    horizons_sec=HORIZONS_SEC,
    thresholds_ticks=THRESHOLDS_TICKS,
    adverse_ticks=ADVERSE_TICKS,
    sampling_step_rows=SAMPLING_STEP_ROWS,
    drop_last_incomplete=DROP_LAST_INCOMPLETE,
)

# Смотрим компактную сводку по бинарным таргетам и excursion-метрикам.
summary = summarize_targets(df_targets)
display(df_targets.head())
display(summary["binary_target_share"].head(20))
display(summary["excursion_stats"].head(20))

# Сохраняем результат в parquet.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_targets.to_parquet(OUTPUT_PATH, index=False)
print("Сохранено в", OUTPUT_PATH)


D:\tumar\okx-hft-research\research\target_building\build_targets_dataset.py:348: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col_name] = values
D:\tumar\okx-hft-research\research\target_building\build_targets_dataset.py:348: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col_name] = values
D:\tumar\okx-hft-research\research\target_building\build_targets_dataset.py:348: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consid

,ts_event,inst_id,anchor_snapshot_ts,reconstruction_mode,mid_px,spread_px,long_max_favorable_excursion_ticks_within_30s,long_max_adverse_excursion_ticks_within_30s,short_max_favorable_excursion_ticks_within_30s,short_max_adverse_excursion_ticks_within_30s,...,long_hit_up_100t_before_down_50t_within_180s,short_hit_down_100t_before_up_50t_within_180s,long_hit_up_150t_before_down_75t_within_180s,short_hit_down_150t_before_up_75t_within_180s,long_hit_up_200t_before_down_100t_within_180s,short_hit_down_200t_before_up_100t_within_180s,long_hit_up_260t_before_down_120t_within_180s,short_hit_down_260t_before_up_120t_within_180s,long_hit_up_300t_before_down_150t_within_180s,short_hit_down_300t_before_up_150t_within_180s
0,2026-03-23 00:00:00+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,1642.0,0.0,0.0,1642.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1,2026-03-23 00:00:00.100000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,1642.0,0.0,0.0,1642.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,2026-03-23 00:00:00.200000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,1642.0,0.0,0.0,1642.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
3,2026-03-23 00:00:00.300000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67840.65,0.1,1544.0,0.0,0.0,1544.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,2026-03-23 00:00:00.400000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67843.85,0.1,1512.0,0.0,0.0,1512.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


,target,share_of_ones
0,long_hit_up_100t_before_down_50t_within_120s,0.406482
1,long_hit_up_100t_before_down_50t_within_180s,0.406798
2,long_hit_up_100t_before_down_50t_within_30s,0.381029
3,long_hit_up_100t_before_down_50t_within_60s,0.402733
4,long_hit_up_100t_before_down_within_120s,0.497051
5,long_hit_up_100t_before_down_within_180s,0.497925
6,long_hit_up_100t_before_down_within_30s,0.449442
7,long_hit_up_100t_before_down_within_60s,0.488477
8,long_hit_up_100t_within_120s,0.772028
9,long_hit_up_100t_within_180s,0.814060


,target,mean,std,min,median,max
0,long_max_favorable_excursion_ticks_within_30s,208.719998,332.487358,0.0,126.0,14245.0
1,long_max_adverse_excursion_ticks_within_30s,204.362299,267.117191,0.0,126.0,6059.0
2,short_max_favorable_excursion_ticks_within_30s,204.362299,267.117191,0.0,126.0,6059.0
3,short_max_adverse_excursion_ticks_within_30s,208.719998,332.487358,0.0,126.0,14245.0
4,long_max_favorable_excursion_ticks_within_60s,320.062928,473.332931,0.0,206.0,14245.0
5,long_max_adverse_excursion_ticks_within_60s,310.213077,375.396816,0.0,202.0,7774.0
6,short_max_favorable_excursion_ticks_within_60s,310.213077,375.396816,0.0,202.0,7774.0
7,short_max_adverse_excursion_ticks_within_60s,320.062928,473.332931,0.0,206.0,14245.0
8,long_max_favorable_excursion_ticks_within_120s,481.471091,682.668032,0.0,321.0,25463.0
9,long_max_adverse_excursion_ticks_within_120s,460.234404,527.722588,0.0,312.0,7774.0


Сохранено в D:\tumar\okx-hft-research\outputs\targets\btc_usdt_swap_targets_dataset.parquet


In [6]:
# Пример запуска модуля из командной строки.
cli_example = r'''
python -m research.target_building.build_targets_dataset \
  --parquet-path data/processed/modeling/btc_usdt_swap_model_dataset_100ms_baseline_20260323_000000__20260325_235959.parquet \
  --output-path outputs/targets/btc_usdt_swap_targets_dataset.parquet \
  --horizons-sec 30,60,120,180 \
  --thresholds-ticks 50,100,150,200,260,300 \
  --adverse-ticks 25,50,75,100,120,150
'''
print(cli_example)



python -m research.target_building.build_targets_dataset \
  --parquet-path data/processed/modeling/btc_usdt_swap_model_dataset_100ms_baseline_20260323_000000__20260325_235959.parquet \
  --output-path outputs/targets/btc_usdt_swap_targets_dataset.parquet \
  --horizons-sec 30,60,120,180 \
  --thresholds-ticks 50,100,150,200,260,300 \
  --adverse-ticks 25,50,75,100,120,150

